In [0]:
from datetime import datetime
import uuid

run_id = "uc16_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")
task_name = "build_appointments_clean"

spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'STARTED',
  current_timestamp(),
  NULL,
  NULL,
  NULL
)
""")

In [0]:
%sql
CREATE OR REPLACE TABLE poc_health_no_show.silver.patient_history_features
COMMENT 'Historical appointment and no-show behavior calculated only from appointments that occurred before the current appointment.'
AS
SELECT
  current_appointment.appointment_id,
  current_appointment.patient_pseudo_id,
  current_appointment.appointment_date,
  COUNT(history.appointment_id) AS prior_appointments,
  COALESCE(SUM(CASE WHEN history.no_show_flag THEN 1 ELSE 0 END), 0) AS prior_no_shows,
  CASE
    WHEN COUNT(history.appointment_id) = 0 THEN 0.0
    ELSE SUM(CASE WHEN history.no_show_flag THEN 1 ELSE 0 END)
      / COUNT(history.appointment_id)
  END AS prior_no_show_rate
FROM poc_health_no_show.silver.appointments_clean current_appointment
LEFT JOIN poc_health_no_show.silver.appointments_clean history
  ON current_appointment.patient_pseudo_id = history.patient_pseudo_id
 AND history.appointment_date < current_appointment.appointment_date
 AND history.data_quality_status = 'VALID'
WHERE current_appointment.data_quality_status = 'VALID'
GROUP BY
  current_appointment.appointment_id,
  current_appointment.patient_pseudo_id,
  current_appointment.appointment_date;

In [0]:
%sql
SELECT
  CASE
    WHEN prior_appointments = 0 THEN 'NEW_PATIENT'
    WHEN prior_no_show_rate >= 0.25 THEN 'HIGH_HISTORY_RISK'
    WHEN prior_no_show_rate >= 0.10 THEN 'MEDIUM_HISTORY_RISK'
    ELSE 'LOW_HISTORY_RISK'
  END AS history_profile,
  COUNT(*) AS appointment_count,
  ROUND(AVG(CASE WHEN no_show_flag THEN 1.0 ELSE 0.0 END) * 100, 2) AS no_show_rate_pct
FROM poc_health_no_show.silver.appointments_clean a
JOIN poc_health_no_show.silver.patient_history_features h
  ON a.appointment_id = h.appointment_id
GROUP BY history_profile
ORDER BY appointment_count DESC;

In [0]:
spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'SUCCESS',
  current_timestamp(),
  current_timestamp(),
  NULL,
  NULL
)
""")